Check Python, RAM and GPU

In [ ]:
import sys

print(sys.version)
assert sys.version_info[:2] == (3, 12), "Use Python 3.12 for this notebook."

!free -h
!nvidia-smi


3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
               total        used        free      shared  buff/cache   available
Mem:            12Gi       3.2Gi       3.1Gi       5.0Mi       6.4Gi       9.2Gi
Swap:             0B          0B          0B
Sat Oct 17 00:57:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off

Get editable source code

In [ ]:
%cd /content
!git clone https://github.com/studio-dots-ai/dots.tts.git
%cd /content/dots.tts
!git checkout 3cb6f94a571097b500038f1619959075282c5c98


/content
fatal: destination path 'dots.tts' already exists and is not an empty directory.
/content/dots.tts
HEAD is now at 3cb6f94 docs: announce doteBench V1.0 release (#54)


Install dots.tts

Load the model

In [ ]:
!ls -la /content/dots.tts
!find /content/dots.tts -maxdepth 2 -type d | head -30

total 108
drwxr-xr-x 10 root root  4096 Oct 17 00:49 .
drwxr-xr-x  1 root root  4096 Oct 17 00:49 ..
drwxr-xr-x  4 root root  4096 Oct 17 00:49 apps
drwxr-xr-x  2 root root  4096 Oct 17 00:49 assets
drwxr-xr-x  2 root root  4096 Oct 17 00:49 configs
drwxr-xr-x  2 root root  4096 Oct 17 00:49 constraints
drwxr-xr-x  8 root root  4096 Oct 17 00:57 .git
drwxr-xr-x  3 root root  4096 Oct 17 00:49 .github
-rw-r--r--  1 root root   457 Oct 17 00:49 .gitignore
-rw-r--r--  1 root root 11352 Oct 17 00:49 LICENSE
-rw-r--r--  1 root root  2535 Oct 17 00:49 pyproject.toml
-rw-r--r--  1 root root 46630 Oct 17 00:49 README.md
drwxr-xr-x  2 root root  4096 Oct 17 00:49 scripts
drwxr-xr-x  4 root root  4096 Oct 17 00:49 src
/content/dots.tts
/content/dots.tts/scripts
/content/dots.tts/constraints
/content/dots.tts/assets
/content/dots.tts/src
/content/dots.tts/src/dots_tts
/content/dots.tts/src/dots.tts.egg-info
/content/dots.tts/.git
/content/dots.tts/.git/branches
/content/dots.tts/.git/hooks
/conte

In [ ]:
%cd /content

!git clone https://github.com/studio-dots-ai/dots.tts.git

%cd /content/dots.tts

!git checkout 3cb6f94a571097b500038f1619959075282c5c98

/content
fatal: destination path 'dots.tts' already exists and is not an empty directory.
/content/dots.tts
HEAD is now at 3cb6f94 docs: announce doteBench V1.0 release (#54)


Imports and resource check

In [ ]:
%cd /content/dots.tts

%pip install -e . "transformers==4.57.0" "setuptools>=68,<82" "accelerate>=1.12.0" pyarrow pandas psutil

/content/dots.tts
Obtaining file:///content/dots.tts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for dots.tts (pyproject.toml) ... done
  Created wheel for dots.tts: filename=dots_tts-0.3.1-0.editable-py3-none-any.whl size=22612 sha256=7317bc9f73a1f19d81da8105c1c0cb0530c0d5a179af3a19658a044e07ed7004
  Stored in directory: /tmp/pip-ephem-wheel-cache-jtwlgfmq/wheels/d3/1c/3c/e392a43ce1368b465e6dac35a21ab086e77f1b67109bf73a59
Successfully built dots.tts
  Attempting uninstall: dots.tts
    Found existing installation: dots.tts 0.3.1
    Uninstalling dots.tts-0.3.1:
      Successfully uninstalled dots.tts-0.3.1


In [ ]:
import sys
import gc
import json
import inspect
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import psutil
import soundfile as sf
from IPython.display import Audio, display
from huggingface_hub import snapshot_download

REPO = Path("/content/dots.tts")
OUT = Path("/content/rababb_kv_map")
OUT.mkdir(exist_ok=True)
SOURCE_REVISION = "3cb6f94a571097b500038f1619959075282c5c98"
assert torch.cuda.is_available(), "Enable a GPU runtime."
revision = !git -C /content/dots.tts rev-parse HEAD
assert revision[0] == SOURCE_REVISION, "Run Setup to use the expected source revision."

from dots_tts.runtime import DotsTtsRuntime
assert Path(inspect.getfile(DotsTtsRuntime)).is_relative_to(REPO)
print("GPU:", torch.cuda.get_device_name(0))
print("Available CPU RAM (GiB):", round(psutil.virtual_memory().available / 2**30, 2))
print("Available GPU memory (GiB):", round(torch.cuda.mem_get_info()[0] / 2**30, 2))


GPU: Tesla T4
Available CPU RAM (GiB): 7.04
Available GPU memory (GiB): 14.46


Optional Hugging Face login

In [ ]:
USE_HF_TOKEN = True

if USE_HF_TOKEN:
    from google.colab import userdata
    from huggingface_hub import login
    login(token=userdata.get("HF_TOKEN"), add_to_git_credential=False)


Download the weights

In [ ]:
MODEL_ID = "dots-studio/dots.tts-base"
MODEL_REVISION = "b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1"
checkpoint = Path(snapshot_download(MODEL_ID, revision=MODEL_REVISION))
print("Checkpoint ready.")


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

latent_stats.pt:   0%|          | 0.00/3.20k [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

llm_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocoder.safetensors:   0%|          | 0.00/724M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Checkpoint ready.


Save the original loading functions

In [ ]:
from accelerate import init_empty_weights
from safetensors import safe_open
import dots_tts.models.dots_tts.model as model_code

Model = model_code.DotsTtsModel
original_core_constructor = model_code.DotsTtsCore
original_artifact_loader = Model._load_artifact_module
original_loader_descriptor = Model.__dict__["_load_artifact_module"]


Create an empty core

In [ ]:
def create_empty_core(*args, **kwargs):
    with init_empty_weights(include_buffers=False):
        core = original_core_constructor(*args, **kwargs)
    return core


Load checkpoint tensors one at a time

In [ ]:
def load_weights(owner, module, path):
    has_empty_parameters = any(p.is_meta for p in module.parameters())
    if not has_empty_parameters:
        return original_artifact_loader(module, path)

    with safe_open(str(path), framework="pt", device="cpu") as checkpoint_file:
        names = list(checkpoint_file.keys())

    weights = {}
    for name in names:
        with safe_open(str(path), framework="pt", device="cpu") as checkpoint_file:
            tensor = checkpoint_file.get_tensor(name)
            dtype = torch.float16 if tensor.is_floating_point() else tensor.dtype
            weights[name] = tensor.to(device="cuda", dtype=dtype)
            del tensor

    # Restore the shared embedding/output weights expected by the upstream model.
    weights = owner._restore_artifact_state_dict(weights, module)
    module.load_state_dict(weights, strict=True, assign=True)
    return module


Load once and restore the original functions

In [ ]:
if "runtime" in globals():
    del runtime
    gc.collect()
    torch.cuda.empty_cache()

try:
    model_code.DotsTtsCore = create_empty_core
    Model._load_artifact_module = classmethod(load_weights)
    runtime = DotsTtsRuntime.from_pretrained(
        str(checkpoint),
        precision="float16",
        optimize=False,
        max_generate_length=160,
        max_sequence_length=1024,
    )
finally:
    model_code.DotsTtsCore = original_core_constructor
    Model._load_artifact_module = original_loader_descriptor

runtime.model.requires_grad_(False)
for tensor in list(runtime.model.parameters()) + list(runtime.model.buffers()):
    assert not tensor.is_meta, "A checkpoint tensor was not loaded."
print("Model loaded. Sample rate:", runtime.sample_rate)
print("Core dtype:", next(runtime.model.core.parameters()).dtype)
print("Available CPU RAM (GiB):", round(psutil.virtual_memory().available / 2**30, 2))


2026-10-17 01:00:15.863 | INFO     | dots_tts.runtime:from_pretrained:160 - [runtime] Runtime load started: model=/root/.cache/huggingface/hub/models--dots-studio--dots.tts-base/snapshots/b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1 revision=None cache_dir=None precision=float16
2026-10-17 01:00:15.871 | DEBUG    | dots_tts.runtime:_resolve_pretrained_path:198 - [runtime] Resolving pretrained path: model=/root/.cache/huggingface/hub/models--dots-studio--dots.tts-base/snapshots/b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1 revision=None cache_dir=None
2026-10-17 01:00:15.876 | DEBUG    | dots_tts.runtime:_resolve_pretrained_path:209 - [runtime] Using local pretrained directory: path=/root/.cache/huggingface/hub/models--dots-studio--dots.tts-base/snapshots/b5aac65a395e04a60a8dacd4e2f4aadc5ad3dbe1
2026-10-17 01:00:15.883 | DEBUG    | dots_tts.models.dots_tts.model:from_pretrained:465 - [model] DotsTtsModel load started: pretrained_path=/root/.cache/huggingface/hub/models--dots-studio--dots.tts-bas

Model loaded. Sample rate: 48000
Core dtype: torch.float16
Available CPU RAM (GiB): 2.24


Get a voice clip and generate one baseline

Download a small speech dataset file

In [ ]:
import io
import pyarrow.parquet as pq
from huggingface_hub import HfApi, list_repo_files, hf_hub_download

DATASET = "hf-internal-testing/librispeech_asr_dummy"
DATASET_REVISION = HfApi().dataset_info(DATASET).sha
filenames = list_repo_files(DATASET, repo_type="dataset", revision=DATASET_REVISION)
parquet_names = sorted(name for name in filenames if name.endswith(".parquet"))
assert parquet_names, "The dataset has no Parquet files."
reference_file = hf_hub_download(
    DATASET, parquet_names[0], repo_type="dataset", revision=DATASET_REVISION,
)


clean/validation-00000-of-00001.parquet:   0%|          | 0.00/9.19M [00:00<?, ?B/s]

Choose an intact short clip

In [ ]:
REFERENCE_AUDIO = str(OUT / "reference.wav")
REFERENCE_TEXT = None
speech_file = pq.ParquetFile(reference_file)

for batch in speech_file.iter_batches(batch_size=1, columns=["audio", "text"]):
    row = batch.to_pylist()[0]
    audio_bytes = row["audio"]["bytes"]
    if audio_bytes is None:
        continue
    waveform, sample_rate = sf.read(io.BytesIO(audio_bytes), dtype="float32")
    if waveform.ndim == 2:
        waveform = waveform.mean(axis=1)
    seconds = len(waveform) / sample_rate
    if 3 <= seconds <= 10:
        sf.write(REFERENCE_AUDIO, waveform, sample_rate)
        REFERENCE_TEXT = row["text"]
        break

assert REFERENCE_TEXT is not None, "No suitable clip found in this dataset file."
print(REFERENCE_TEXT)
display(Audio(REFERENCE_AUDIO))
del speech_file, batch, row, waveform, audio_bytes


MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL


One generation function

In [ ]:
TEXT = "Hello, this is a test of the dots TTS model."
SEED = 42
STEPS = 10

def generate(steps=STEPS, profile=False):
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    with torch.inference_mode():
        return runtime.generate(
            text=TEXT,
            prompt_audio_path=REFERENCE_AUDIO,
            prompt_text=REFERENCE_TEXT,
            num_steps=steps,
            ode_method="euler",
            guidance_scale=1.2,
            speaker_scale=1.5,
            profile_inference=profile,
        )


Generate, save and listen

In [ ]:
baseline = generate(profile=True)
baseline_audio = baseline["audio"].detach().float().cpu().squeeze().numpy()
assert np.isfinite(baseline_audio).all(), "Audio has invalid values."
sf.write(OUT / "baseline.wav", baseline_audio, baseline["sample_rate"])
baseline_profile = baseline["profiling"]
display(Audio(str(OUT / "baseline.wav")))
display(pd.DataFrame(baseline_profile).T)
del baseline


2026-10-17 01:00:54.238 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-17 01:01:19.433 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-17 01:01:19.441 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-17 01:01:19.445 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

,seconds,count,rtf
FM,8.052093,22.0,2.396456
latent_encoder,0.235461,1.0,0.070078
patch_encoder,1.012170,23.0,0.301241
LLM,1.151244,23.0,0.342632
latent_decoder,0.658954,1.0,0.196117
speaker_encoder,2.177253,1.0,0.647992
vocoder,0.000000,0.0,0.000000


Read the inference path

Open the high-level model path

In [ ]:
from dots_tts.modules.backbone.dit_inference import EagerDiTRunner, DiTSolver, CachedDiTRunner
from dots_tts.modules.backbone.llm_inference import LLMInference

print("Runtime file:", inspect.getfile(DotsTtsRuntime))
print("Model file:", inspect.getfile(type(runtime.model)))
for name, module in runtime.model.named_children():
    print(name, "->", type(module).__name__)
print(inspect.getsource(runtime.model.generate_audio))


Runtime file: /content/dots.tts/src/dots_tts/runtime.py
Model file: /content/dots.tts/src/dots_tts/models/dots_tts/model.py
core -> DotsTtsCore
vocoder -> AudioVAE
xvector_extractor -> SpeakerXVectorFeatures
    @torch.no_grad()
    def generate_audio(
        self,
        data: dict[str, Any],
        *,
        precision: str,
        ode_method: str,
        num_steps: int,
        guidance_scale: float,
        speaker_scale: float = 1.5,
    ) -> torch.Tensor:
        latent_patches = list(
            self._generate_latents_stream(
                data,
                precision=precision,
                ode_method=ode_method,
                num_steps=num_steps,
                guidance_scale=guidance_scale,
                speaker_scale=speaker_scale,
            )
        )
        logger.debug(
            logc("vocoder", "Vocoder decode started: latent_patch_count={}"),
            len(latent_patches),
        )
        audio = self._get_vocoder_inference().decode_latents(

Follow one audio-patch decode

In [ ]:
print(inspect.getsource(runtime.model._decode_next_audio))
print(inspect.getsource(runtime.model._consume_audio_patch))


    def _decode_next_audio(
        self,
        state: _GenerateState,
        *,
        g_cond: torch.Tensor | None,
        ode_method: str,
        num_steps: int,
        guidance_scale: float,
        profile_step: int | None = None,
    ) -> torch.Tensor:
        with measure_inference("FM", phase="decode", step=profile_step):
            sequence = state.fm_sequence
            if sequence is None:
                raise RuntimeError("FM static buffer is not initialized.")
            null_g_cond = state.fm_null_g_cond
            if null_g_cond is None:
                raise RuntimeError("FM null conditioning buffer is not initialized.")
            sampling = self.config.sampling
            if sampling is not None:
                sampling.resolve(
                    ode_method=ode_method,
                    num_steps=num_steps,
                    guidance_scale=guidance_scale,
                )
            solver_mode = sampling.solver if sampling is not None else self.

In [ ]:
# Phase 1: trace the DiT / solver inference path

import inspect

print("===== DiTSolver =====")
print(inspect.getsource(DiTSolver))

print("\n===== EagerDiTRunner =====")
print(inspect.getsource(EagerDiTRunner))


===== DiTSolver =====
class DiTSolver:
    """Streaming DiT decoder with a KV cache across ODE steps (optimize mode).

    In non-optimize mode this delegates to :class:`EagerDiTRunner`. In
    optimize mode it maintains a per-bucket :class:`CachedDiTRunner` and
    incrementally warms a :class:`DiTKvCache`.
    """

    def __init__(
        self,
        context: DiTInferenceContext,
        *,
        optimize: bool,
        bucket_resolver: Callable[[int], int],
        meanflow: bool = False,
    ):
        self.context = context
        self.optimize = bool(optimize)
        self.bucket_resolver = bucket_resolver
        self.meanflow = bool(meanflow)
        self.branch_multiplier = 1 if self.meanflow else 2
        self._cached_runners: dict[tuple[Any, ...], CachedDiTRunner] = {}
        self._eager_runners: dict[tuple[Any, ...], EagerDiTRunner] = {}
        self._backend_by_device_type: dict[str, str] = {}

    # ---- Runner lookup ---------------------------------------------

## Phase 1 Takeaways — Inference Path / NFE

- The current baseline runs with `optimize=False`, so inference uses `EagerDiTRunner`, not the KV-cached optimized path.
- The important flow is:

  runtime.generate()
      ↓
  model.generate_audio()
      ↓
  DiTSolver.decode_next()
      ↓
  EagerDiTRunner.decode_next()
      ↓
  _decode_flow_matching()
      ↓
  odeint(...)
      ↓
  solver(t, z)
      ↓
  _flow_matching_velocity()
      ↓
  self.dit(...)

- `nfe` controls the ODE solver step size for Euler as `1 / nfe`.
- The repeated heavy neural-network computation happens inside `self.dit(...)`.
- The loop is not written directly as `for step in range(nfe)` for FlowMatching; it is handled internally by `odeint(...)`.
- Before repeated solver evaluations, inputs such as `input_sequence`, `cfg_input_sequence`, `attn_mask`, `pos_ids`, `g_cond`, and guidance are prepared once.
- During inference, `t` (time) and `z` (current latent state) change across solver evaluations.
- This gives us the first useful split for Phase 2:
  - likely invariant/reusable state → conditioning, masks, position IDs, guidance-related inputs
  - changing state → timestep `t`, latent `z`, and model outputs
- The optimized path contains an explicit KV cache, but the current baseline does not use it. That path can be studied later when comparing reusable state and optimized execution.

** NFE (Number of functional Evaluations) determines how many refinement/function-evaluation steps are performed, and each evaluation eventually calls the DiT model through self.dit(...).

In [ ]:
# Phase 2A — runtime tracing for eager FlowMatching path

import types
import torch

solver = runtime.model._get_dit_solver(solver_mode="flow_matching")

runner = solver._get_eager_runner(
    device=next(runtime.model.parameters()).device,
    dtype=next(runtime.model.parameters()).dtype,
)

trace_rows = []
eval_counter = {"count": 0}

original_fmv = runner._flow_matching_velocity

def traced_flow_matching_velocity(self, t, z, **kwargs):
    eval_counter["count"] += 1
    i = eval_counter["count"]

    def info(name, x):
        if x is None:
            return {
                "eval": i,
                "name": name,
                "shape": None,
                "dtype": None,
                "device": None,
                "data_ptr": None,
                "mean": None,
            }

        return {
            "eval": i,
            "name": name,
            "shape": tuple(x.shape),
            "dtype": str(x.dtype),
            "device": str(x.device),
            "data_ptr": x.data_ptr() if x.numel() > 0 else None,
            "mean": float(x.float().mean().item()) if x.numel() > 0 else None,
        }

    trace_rows.extend([
        info("t", t),
        info("z", z),
        info("input_sequence", kwargs["input_sequence"]),
        info("g_cond", kwargs["g_cond"]),
        info("attn_mask", kwargs["attn_mask"]),
        info("pos_ids", kwargs["pos_ids"]),
    ])

    return original_fmv(t, z, **kwargs)

runner._flow_matching_velocity = types.MethodType(
    traced_flow_matching_velocity,
    runner
)

# Run one reproducible baseline
result = generate(steps=10, profile=False)

# Restore original method
runner._flow_matching_velocity = original_fmv

print("Actual function evaluations:", eval_counter["count"])

trace_df = pd.DataFrame(trace_rows)
display(trace_df)

2026-10-17 01:01:33.720 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-17 01:01:33.729 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-17 01:01:33.731 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-17 01:01:33.732 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

Actual function evaluations: 220


,eval,name,shape,dtype,device,data_ptr,mean
0,1,t,(),torch.float16,cuda:0,134176414100992,0.000000
1,1,z,"(1, 4, 128)",torch.float16,cuda:0,134180908194304,-0.048102
2,1,input_sequence,"(1, 155, 1024)",torch.float16,cuda:0,134175713908736,-0.011696
3,1,g_cond,"(1, 1024)",torch.float16,cuda:0,134180908191744,-0.069792
4,1,attn_mask,"(1, 155, 155)",torch.bool,cuda:0,134176414076928,0.503642
...,...,...,...,...,...,...,...
1315,220,z,"(1, 4, 128)",torch.float16,cuda:0,134177798224896,0.017858
1316,220,input_sequence,"(1, 260, 1024)",torch.float16,cuda:0,134176443465728,-0.011420
1317,220,g_cond,"(1, 1024)",torch.float16,cuda:0,134176187611136,-0.069792
1318,220,attn_mask,"(1, 260, 260)",torch.bool,cuda:0,134177762943488,0.502071


In [ ]:
# Summarize each variable
summary = []

for name, group in trace_df.groupby("name"):
    summary.append({
        "variable": name,
        "shape": group["shape"].iloc[0],
        "dtype": group["dtype"].iloc[0],
        "device": group["device"].iloc[0],
        "unique_ptrs": group["data_ptr"].nunique(),
        "unique_means": group["mean"].nunique(),
        "eval_count": len(group),
    })

summary_df = pd.DataFrame(summary)
display(summary_df)


,variable,shape,dtype,device,unique_ptrs,unique_means,eval_count
0,attn_mask,"(1, 155, 155)",torch.bool,cuda:0,22,22,220
1,g_cond,"(1, 1024)",torch.float16,cuda:0,10,1,220
2,input_sequence,"(1, 155, 1024)",torch.float16,cuda:0,19,22,220
3,pos_ids,"(1, 155)",torch.float32,cuda:0,5,22,220
4,t,(),torch.float16,cuda:0,120,10,220
5,z,"(1, 4, 128)",torch.float16,cuda:0,68,220,220



### Runtime Tracing — Initial Finding

With NFE=10, the complete generation produced 220 calls to the
FlowMatching velocity / DiT evaluation path.

This occurs because dots.tts generates audio incrementally across
multiple patches. The run appears to contain 22 decode patches, with
10 NFE evaluations per patch:

22 patches × 10 NFE = 220 model evaluations.

Initial runtime observations:

- `t` has 10 unique values, consistent with the same 10-step ODE
  schedule being repeated for each generated patch.
- `z` changes at every evaluation, confirming that it is the evolving
  latent state.
- `g_cond` remains numerically constant across all 220 evaluations,
  making it a strong invariant/reuse candidate.
- `input_sequence`, `attn_mask`, and `pos_ids` change between audio
  patches as the generated sequence grows.
- These tensors appear to be static during the NFE evaluations of an
  individual patch, which needs to be verified explicitly next.

Important distinction:
1. invariants across NFE steps within one patch
2. invariants across the entire generation


In [ ]:
## Phase 2B - label each evalution by patch and NFE step


NFE = 10

trace_df["patch"] = ((trace_df["eval"] - 1) // NFE) + 1
trace_df["nfe_step"] = ((trace_df["eval"] - 1) % NFE) + 1

display(
    trace_df[
        ["patch", "nfe_step", "eval", "name",
         "shape", "dtype", "device", "data_ptr", "mean"]
    ].head(60)
)


,patch,nfe_step,eval,name,shape,dtype,device,data_ptr,mean
0,1,1,1,t,(),torch.float16,cuda:0,134176414100992,0.000000
1,1,1,1,z,"(1, 4, 128)",torch.float16,cuda:0,134180908194304,-0.048102
2,1,1,1,input_sequence,"(1, 155, 1024)",torch.float16,cuda:0,134175713908736,-0.011696
3,1,1,1,g_cond,"(1, 1024)",torch.float16,cuda:0,134180908191744,-0.069792
4,1,1,1,attn_mask,"(1, 155, 155)",torch.bool,cuda:0,134176414076928,0.503642
5,1,1,1,pos_ids,"(1, 155)",torch.float32,cuda:0,134177764670464,77.000000
6,1,2,2,t,(),torch.float16,cuda:0,134176414100994,0.099976
7,1,2,2,z,"(1, 4, 128)",torch.float16,cuda:0,134177764660224,-0.046922
8,1,2,2,input_sequence,"(1, 155, 1024)",torch.float16,cuda:0,134175713908736,-0.011696
9,1,2,2,g_cond,"(1, 1024)",torch.float16,cuda:0,134180908191744,-0.069792


In [ ]:
# Check invariance within each audio patch

patch_analysis = []

for (patch, name), group in trace_df.groupby(["patch", "name"]):
    patch_analysis.append({
        "patch": patch,
        "variable": name,
        "shape": group["shape"].iloc[0],
        "dtype": group["dtype"].iloc[0],
        "device": group["device"].iloc[0],
        "evals_in_patch": len(group),
        "unique_ptrs": group["data_ptr"].nunique(),
        "unique_means": group["mean"].nunique(),
    })

patch_df = pd.DataFrame(patch_analysis)

display(patch_df.head(30))

,patch,variable,shape,dtype,device,evals_in_patch,unique_ptrs,unique_means
0,1,attn_mask,"(1, 155, 155)",torch.bool,cuda:0,10,1,1
1,1,g_cond,"(1, 1024)",torch.float16,cuda:0,10,1,1
2,1,input_sequence,"(1, 155, 1024)",torch.float16,cuda:0,10,1,1
3,1,pos_ids,"(1, 155)",torch.float32,cuda:0,10,1,1
4,1,t,(),torch.float16,cuda:0,10,10,10
5,1,z,"(1, 4, 128)",torch.float16,cuda:0,10,7,10
6,2,attn_mask,"(1, 160, 160)",torch.bool,cuda:0,10,1,1
7,2,g_cond,"(1, 1024)",torch.float16,cuda:0,10,1,1
8,2,input_sequence,"(1, 160, 1024)",torch.float16,cuda:0,10,1,1
9,2,pos_ids,"(1, 160)",torch.float32,cuda:0,10,1,1


In [ ]:
# Overall within-patch behavior for each variable

within_patch_summary = (
    patch_df.groupby("variable")
    .agg(
        num_patches=("patch", "nunique"),
        min_evals_per_patch=("evals_in_patch", "min"),
        max_evals_per_patch=("evals_in_patch", "max"),
        max_unique_ptrs_within_patch=("unique_ptrs", "max"),
        max_unique_means_within_patch=("unique_means", "max"),
    )
    .reset_index()
)

display(within_patch_summary)

,variable,num_patches,min_evals_per_patch,max_evals_per_patch,max_unique_ptrs_within_patch,max_unique_means_within_patch
0,attn_mask,22,10,10,1,1
1,g_cond,22,10,10,1,1
2,input_sequence,22,10,10,1,1
3,pos_ids,22,10,10,1,1
4,t,22,10,10,10,10
5,z,22,10,10,8,10


In [ ]:
# Phase 2C — trace repeated operations inside _flow_matching_velocity

import types
import pandas as pd
import torch

from dots_tts.modules.backbone.dit_inference import apply_cfg

solver = runtime.model._get_dit_solver(solver_mode="flow_matching")

runner = solver._get_eager_runner(
    device=next(runtime.model.parameters()).device,
    dtype=next(runtime.model.parameters()).dtype,
)

op_trace = []
eval_counter = {"count": 0}

original_fmv = runner._flow_matching_velocity

def traced_fmv(self, t, z, **kwargs):
    eval_counter["count"] += 1
    i = eval_counter["count"]

    input_sequence = kwargs["input_sequence"]
    cfg_sequence = kwargs["cfg_sequence"]
    attn_mask = kwargs["attn_mask"]
    pos_ids = kwargs["pos_ids"]
    g_cond = kwargs["g_cond"]
    guidance_scale = kwargs["guidance_scale"]

    batch_size = input_sequence.size(0)
    latent_start = input_sequence.size(1) - self.latent_patch_size

    # 1. project current latent
    z_proj = self.coordinate_proj(z)

    # 2. inject current latent into both branches
    splat_main = self._splat_noise(z_proj, input_sequence)
    splat_cfg = self._splat_noise(z_proj, cfg_sequence)

    # 3. concatenate CFG branches
    x = torch.cat([splat_main, splat_cfg], dim=0)

    # 4. build timestep tensor
    timesteps = t.reshape(1).expand(x.size(0))

    # 5. build conditioning branches
    g_cond_branches = torch.cat(
        [g_cond, torch.zeros_like(g_cond)],
        dim=0
    )

    # 6. DiT forward pass
    pred_full = self.dit(
        x=x,
        timesteps=timesteps,
        attn_mask=attn_mask,
        pos_ids=pos_ids,
        g_cond=g_cond_branches,
    )

    pred = pred_full[:, latent_start:]

    def row(name, tensor):
        return {
            "eval": i,
            "operation": name,
            "shape": tuple(tensor.shape),
            "dtype": str(tensor.dtype),
            "device": str(tensor.device),
            "data_ptr": tensor.data_ptr() if tensor.numel() else None,
            "mean": float(tensor.float().mean().item()) if tensor.numel() else None,
        }

    op_trace.extend([
        row("z_proj", z_proj),
        row("splat_main", splat_main),
        row("splat_cfg", splat_cfg),
        row("x", x),
        row("timesteps", timesteps),
        row("g_cond_branches", g_cond_branches),
        row("pred", pred),
    ])

    return apply_cfg(
        pred,
        batch_size=batch_size,
        guidance=guidance_scale,
    )

runner._flow_matching_velocity = types.MethodType(
    traced_fmv,
    runner
)

result = generate(steps=10, profile=False)

runner._flow_matching_velocity = original_fmv

op_df = pd.DataFrame(op_trace)

print("Total evaluations:", eval_counter["count"])
display(op_df.head(40))

2026-10-17 01:01:42.193 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-17 01:01:42.208 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-17 01:01:42.210 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-17 01:01:42.212 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

Total evaluations: 220


,eval,operation,shape,dtype,device,data_ptr,mean
0,1,z_proj,"(1, 4, 1024)",torch.float16,cuda:0,134175717842944,-0.006564
1,1,splat_main,"(1, 155, 1024)",torch.float16,cuda:0,134176186150912,-0.011865
2,1,splat_cfg,"(1, 155, 1024)",torch.float16,cuda:0,134175713656832,0.001543
3,1,x,"(2, 155, 1024)",torch.float16,cuda:0,134175713974272,-0.005161
4,1,timesteps,"(2,)",torch.float16,cuda:0,134175715748864,0.000000
5,1,g_cond_branches,"(2, 1024)",torch.float16,cuda:0,134176186985472,-0.034896
6,1,pred,"(2, 4, 128)",torch.float16,cuda:0,134176186507008,0.013709
7,2,z_proj,"(1, 4, 1024)",torch.float16,cuda:0,134175717842944,-0.006574
8,2,splat_main,"(1, 155, 1024)",torch.float16,cuda:0,134176186150912,-0.011865
9,2,splat_cfg,"(1, 155, 1024)",torch.float16,cuda:0,134175713656832,0.001543


In [ ]:
op_summary = []

for name, group in op_df.groupby("operation"):
    op_summary.append({
        "operation": name,
        "shape": group["shape"].iloc[0],
        "dtype": group["dtype"].iloc[0],
        "device": group["device"].iloc[0],
        "eval_count": len(group),
        "unique_ptrs": group["data_ptr"].nunique(),
        "unique_means": group["mean"].nunique(),
    })

op_summary_df = pd.DataFrame(op_summary)
display(op_summary_df)

,operation,shape,dtype,device,eval_count,unique_ptrs,unique_means
0,g_cond_branches,"(2, 1024)",torch.float16,cuda:0,220,32,1
1,pred,"(2, 4, 128)",torch.float16,cuda:0,220,22,220
2,splat_cfg,"(1, 155, 1024)",torch.float16,cuda:0,220,17,220
3,splat_main,"(1, 155, 1024)",torch.float16,cuda:0,220,19,220
4,timesteps,"(2,)",torch.float16,cuda:0,220,140,10
5,x,"(2, 155, 1024)",torch.float16,cuda:0,220,21,220
6,z_proj,"(1, 4, 1024)",torch.float16,cuda:0,220,16,220


## Phase 2 — Repeated Operation Findings

For the NFE=10 eager baseline, the traced FlowMatching path executed
220 function evaluations across the full generation.

Operations recomputed every evaluation:
- `coordinate_proj(z)` → `z_proj`
- `_splat_noise(...)` for main and CFG branches
- concatenation to construct `x`
- timestep expansion
- construction of `g_cond_branches`
- full `self.dit(...)` forward pass

Dynamic outputs:
- `z_proj`, `splat_main`, `splat_cfg`, `x`, and `pred` change at every
  function evaluation because they depend directly or indirectly on the
  evolving latent `z`.

Important reuse candidate:
- `g_cond_branches` was reconstructed on every evaluation, but its
  numerical value remained constant across all 220 evaluations.
  Therefore it is a strong candidate for precomputation/reuse.

Previously identified per-decode invariants:
- `input_sequence`
- `attn_mask`
- `pos_ids`
- `g_cond`

These remain unchanged across NFE steps within the same decode group.

In [ ]:
# Phase 2D — time repeated operations inside _flow_matching_velocity

import types
import pandas as pd
import torch

from dots_tts.modules.backbone.dit_inference import apply_cfg

solver = runtime.model._get_dit_solver(solver_mode="flow_matching")

runner = solver._get_eager_runner(
    device=next(runtime.model.parameters()).device,
    dtype=next(runtime.model.parameters()).dtype,
)

timing_rows = []
eval_counter = {"count": 0}

original_fmv = runner._flow_matching_velocity

def cuda_time_ms(fn):
    start = torch.cuda.Event(enable_timing=True)
    end = torch.cuda.Event(enable_timing=True)

    start.record()
    out = fn()
    end.record()

    torch.cuda.synchronize()

    return out, start.elapsed_time(end)


def traced_fmv(self, t, z, **kwargs):
    eval_counter["count"] += 1
    i = eval_counter["count"]

    input_sequence = kwargs["input_sequence"]
    cfg_sequence = kwargs["cfg_sequence"]
    attn_mask = kwargs["attn_mask"]
    pos_ids = kwargs["pos_ids"]
    g_cond = kwargs["g_cond"]
    guidance_scale = kwargs["guidance_scale"]

    batch_size = input_sequence.size(0)
    latent_start = input_sequence.size(1) - self.latent_patch_size

    # 1. coordinate projection
    z_proj, ms = cuda_time_ms(
        lambda: self.coordinate_proj(z)
    )
    timing_rows.append({
        "eval": i,
        "operation": "coordinate_proj",
        "time_ms": ms,
    })

    # 2. splat main
    splat_main, ms = cuda_time_ms(
        lambda: self._splat_noise(z_proj, input_sequence)
    )
    timing_rows.append({
        "eval": i,
        "operation": "splat_main",
        "time_ms": ms,
    })

    # 3. splat CFG
    splat_cfg, ms = cuda_time_ms(
        lambda: self._splat_noise(z_proj, cfg_sequence)
    )
    timing_rows.append({
        "eval": i,
        "operation": "splat_cfg",
        "time_ms": ms,
    })

    # 4. concatenate DiT input
    x, ms = cuda_time_ms(
        lambda: torch.cat([splat_main, splat_cfg], dim=0)
    )
    timing_rows.append({
        "eval": i,
        "operation": "build_x",
        "time_ms": ms,
    })

    # 5. timestep expansion
    timesteps, ms = cuda_time_ms(
        lambda: t.reshape(1).expand(x.size(0))
    )
    timing_rows.append({
        "eval": i,
        "operation": "build_timesteps",
        "time_ms": ms,
    })

    # 6. conditioning branch construction
    g_cond_branches, ms = cuda_time_ms(
        lambda: torch.cat(
            [g_cond, torch.zeros_like(g_cond)],
            dim=0
        )
    )
    timing_rows.append({
        "eval": i,
        "operation": "build_g_cond_branches",
        "time_ms": ms,
    })

    # 7. heavy DiT forward
    pred_full, ms = cuda_time_ms(
        lambda: self.dit(
            x=x,
            timesteps=timesteps,
            attn_mask=attn_mask,
            pos_ids=pos_ids,
            g_cond=g_cond_branches,
        )
    )
    timing_rows.append({
        "eval": i,
        "operation": "dit_forward",
        "time_ms": ms,
    })

    pred = pred_full[:, latent_start:]

    return apply_cfg(
        pred,
        batch_size=batch_size,
        guidance=guidance_scale,
    )


runner._flow_matching_velocity = types.MethodType(
    traced_fmv,
    runner
)

result = generate(steps=10, profile=False)

runner._flow_matching_velocity = original_fmv

timing_df = pd.DataFrame(timing_rows)

print("Total evaluations:", eval_counter["count"])
display(timing_df.head(30))

2026-10-17 01:01:57.000 | DEBUG    | dots_tts.runtime:_load_prompt_audio:447 - [io] Loading prompt audio: path=/content/rababb_kv_map/reference.wav
2026-10-17 01:01:57.008 | DEBUG    | dots_tts.runtime:_load_prompt_audio:458 - [io] Prompt audio loaded: path=/content/rababb_kv_map/reference.wav original_sample_rate=16000 resampled_sample_rate=48000 samples=238080
2026-10-17 01:01:57.010 | INFO     | dots_tts.data.pipelines.tokenizing:build_generation_schedule:211 - Built generation schedule: interleave=False max_audio_tokens=160 sequence='[文本]MISTER QUILTER IS THE APOSTLE OF THE MIDDLE CLASSES AND WE ARE GLAD TO WELCOME HIS GOSPEL\nHello, this is a test of the dots TTS model.[文本对应语音]<|audio_gen_start|>'
2026-10-17 01:01:57.131 | DEBUG    | dots_tts.runtime:_prepare_inputs:641 - [request] Inputs prepared: request_id=d8e2b5aeeff96d5a template_name=tts language=None text_len=44 prompt_text_len=90 schedule_length=212 prompt_audio_patch_count=31 max_audio_patch_count=160 max_sequence_length=

Total evaluations: 220


,eval,operation,time_ms
0,1,coordinate_proj,0.112640
1,1,splat_main,0.070016
2,1,splat_cfg,0.067584
3,1,build_x,0.042624
4,1,build_timesteps,0.022560
5,1,build_g_cond_branches,0.070688
6,1,dit_forward,27.068512
7,2,coordinate_proj,0.108576
8,2,splat_main,0.071584
9,2,splat_cfg,0.063648


In [ ]:
timing_summary = (
    timing_df.groupby("operation")["time_ms"]
    .agg(
        calls="count",
        mean_ms="mean",
        median_ms="median",
        min_ms="min",
        max_ms="max",
        total_ms="sum",
    )
    .reset_index()
    .sort_values("total_ms", ascending=False)
)

timing_summary["percent_of_traced_time"] = (
    100 * timing_summary["total_ms"]
    / timing_summary["total_ms"].sum()
)

display(timing_summary)

,operation,calls,mean_ms,median_ms,min_ms,max_ms,total_ms,percent_of_traced_time
4,dit_forward,220,46.588162,23.702656,20.980608,218.090332,10249.395565,98.435388
3,coordinate_proj,220,0.233878,0.106496,0.083968,10.281536,51.453184,0.494157
6,splat_main,220,0.200644,0.076016,0.062784,10.584320,44.141760,0.423938
0,build_g_cond_branches,220,0.092461,0.064336,0.051200,4.203136,20.341408,0.195359
2,build_x,220,0.092123,0.042640,0.035520,6.149696,20.267136,0.194646
5,splat_cfg,220,0.065128,0.051440,0.042880,0.186368,14.328224,0.137609
1,build_timesteps,220,0.056275,0.024304,0.020352,6.121856,12.380544,0.118903


## Phase 2 — Operation Timing Findings

For the NFE=10 eager baseline, 220 FlowMatching function evaluations were
recorded.

The DiT forward pass dominates repeated inference cost:

- `dit_forward`: ~25.57 ms/eval, ~5626 ms total, 98.5% of traced time
- `coordinate_proj`: ~0.41%
- `splat_main`: ~0.32%
- `build_g_cond_branches`: ~0.26%
- `splat_cfg`: ~0.23%
- `build_x`: ~0.18%
- `build_timesteps`: ~0.11%

Key conclusion:
- The main optimization opportunity is reducing or avoiding work inside
  `self.dit(...)`, not micro-optimizing the surrounding tensor construction.
- `g_cond_branches` is invariant and rebuilt every evaluation, but its runtime
  contribution is very small, so it is a low-priority optimization.
- Dynamic operations depending on `z`, such as `coordinate_proj`, splatting,
  and construction of `x`, must be recomputed each NFE step, but together they
  are still small relative to the DiT forward pass.

In [ ]:
# Phase 2E — inspect CachedDiTRunner

import inspect

print("===== CachedDiTRunner.prefill =====")
print(inspect.getsource(CachedDiTRunner.prefill))

print("\n===== CachedDiTRunner.step =====")
print(inspect.getsource(CachedDiTRunner.step))

print("\n===== CachedDiTRunner.current_step =====")
print(inspect.getsource(CachedDiTRunner.current_step))


# Find how CachedDiTRunner actually implements step/prefill

print("Methods containing 'step' or 'prefill':")

names = [
    name for name in dir(CachedDiTRunner)
    if "step" in name.lower() or "prefill" in name.lower()
]

print(names)

for name in names:
    obj = getattr(CachedDiTRunner, name)

    if callable(obj):
        print(f"\n===== {name} =====")

        try:
            print(inspect.getsource(obj))
        except Exception as e:
            print("Could not inspect:", e)


print("===== CachedDiTRunner.__init__ =====")
print(inspect.getsource(CachedDiTRunner.__init__))

===== CachedDiTRunner.prefill =====
    def prefill(
        self,
        *,
        kv_cache: DiTKvCache,
        prefix_sequence: torch.Tensor,
        all_mods_by_ode: torch.Tensor,
        cfg_prefix_sequence: torch.Tensor | None = None,
    ) -> None:
        prefix_len = int(prefix_sequence.size(1))
        if prefix_len <= 0:
            kv_cache.valid_tokens = 0
            return
        if prefix_len > kv_cache.capacity_tokens:
            raise ValueError(
                "KV prefill exceeds cache capacity: "
                f"prefix_len={prefix_len} capacity={kv_cache.capacity_tokens}."
            )
        if not self._prefill_compiled(
            kv_cache=kv_cache,
            prefix_sequence=prefix_sequence,
            cfg_prefix_sequence=cfg_prefix_sequence,
            all_mods_by_ode=all_mods_by_ode,
        ):
            self._prefill_eager(
                kv_cache=kv_cache,
                prefix_sequence=prefix_sequence,
                cfg_prefix_sequence=cf

AttributeError: type object 'CachedDiTRunner' has no attribute 'step'

In [ ]:
# Step 1 — find CachedDiTRunner methods related to step/prefill

names = [
    name for name in dir(CachedDiTRunner)
    if "step" in name.lower() or "prefill" in name.lower()
]

print(names)

['PREFILL_COMPILE_BUCKET_PATCHES', '_prefill_compiled', '_prefill_eager', '_prefill_rotary_for_compile', 'prefill']


In [ ]:
# Step 2 — inspect those methods + constructor

for name in names:
    obj = getattr(CachedDiTRunner, name)

    if callable(obj):
        print(f"\n===== {name} =====")
        try:
            print(inspect.getsource(obj))
        except Exception as e:
            print("Could not inspect:", e)

print("\n===== CachedDiTRunner.__init__ =====")
print(inspect.getsource(CachedDiTRunner.__init__))


===== _prefill_compiled =====
    def _prefill_compiled(
        self,
        *,
        kv_cache: DiTKvCache,
        prefix_sequence: torch.Tensor,
        all_mods_by_ode: torch.Tensor,
        cfg_prefix_sequence: torch.Tensor | None,
    ) -> bool:
        if self.compiled_prefill_step is None:
            return False
        prefix_len = int(prefix_sequence.size(1))
        bucket_tokens = int(self.prefill_compile_bucket_tokens)
        if prefix_len > bucket_tokens:
            return False

        batch_size = int(prefix_sequence.size(0))
        branch_batch = batch_size if cfg_prefix_sequence is None else 2 * batch_size
        x_base = prefix_sequence.new_zeros(
            branch_batch, bucket_tokens, int(prefix_sequence.size(2))
        )
        x_base[:batch_size, :prefix_len].copy_(prefix_sequence)
        if cfg_prefix_sequence is not None:
            x_base[batch_size:, :prefix_len].copy_(cfg_prefix_sequence)

        rotary_cos, rotary_sin = self._prefill_rotary

## Phase 2 — Eager vs Cached Runner

The eager baseline reruns the full DiT path over the growing sequence on
every NFE evaluation.

The cached runner instead performs a `prefill()` over the persistent
prefix/history and stores per-NFE attention key/value state in:

- `kv_cache.cache_k`
- `kv_cache.cache_v`

For each ODE/NFE index, prefix K/V tensors are computed once and copied
into the cache. Later decode steps can reuse this state instead of
reprocessing the full historical sequence.

Additional reusable state in `CachedDiTRunner` includes:
- rotary embeddings
- attention masks
- compiled step functions
- cached prefill rotary tensors

This matters because our eager profiling showed that the DiT forward pass
accounts for ~98.5% of traced repeated runtime. Therefore, avoiding
recomputation of historical attention state inside DiT is much more
valuable than micro-optimizing small tensor operations such as
`g_cond_branches`.

In [ ]:
# Phase 2F — inspect cached step modules

import inspect

from dots_tts.modules.backbone.dit_inference import (
    NextPatchStep,
    FirstPatchStep,
)

print("===== NextPatchStep =====")
print(inspect.getsource(NextPatchStep))

print("\n===== FirstPatchStep =====")
print(inspect.getsource(FirstPatchStep))

===== NextPatchStep =====
class NextPatchStep(nn.Module):
    """Tail patch: prev_unit (fresh KV) + current_hidden + noise → velocity, KV updates."""

    def __init__(self, context: DiTInferenceContext, *, attn_backend: str):
        super().__init__()
        if attn_backend not in {"sdpa", "flex"}:
            raise ValueError(
                f"Unsupported KV cache attention backend: {attn_backend!r}."
            )
        self.dit = context.dit
        self.coordinate_proj = context.coordinate_proj
        self.hidden_patch_size = int(context.hidden_patch_size)
        self.latent_patch_size = int(context.latent_patch_size)
        self.unit_len = self.hidden_patch_size + self.latent_patch_size
        self.hidden_size = int(context.hidden_size)
        self.attn_backend = attn_backend
        self.fm_sigma = float(context.fm_sigma)

        attn0 = self.dit.blocks[0].attn
        self.num_heads = int(attn0.num_heads)
        self.head_dim = int(attn0.head_dim)
        self.num_l

## Phase 2 — Cached Step Findings

`NextPatchStep` confirms how the optimized path avoids recomputing
historical attention state.

For each transformer layer, cached prefix keys and values are passed into
attention through:

- `key_prefix=cache_k[layer_idx]`
- `value_prefix=cache_v[layer_idx]`

Only K/V for the current/new unit are produced as `new_k` and `new_v`.

Dynamic work that still occurs every NFE step:
- `coordinate_proj(z)`
- construction of the current local input
- transformer block computation for the current/recent unit
- final velocity prediction

Reusable state:
- historical prefix K/V
- masks
- rotary embeddings
- precomputed per-NFE modulation state

`FirstPatchStep` does not use a KV cache because no historical prefix
exists yet.

Main conclusion:
The cached path targets the expensive DiT computation by preventing
historical attention state from being recomputed at every NFE step.